# 01 — Bronze Layer

**Tujuan:** Mengingesti data mentah `food.parquet` ke Delta Lake as-is, tambah metadata `ingestion_timestamp`.

**Input:** `/home/jovyan/work/food.parquet` (~7GB, 4.487.169 baris)  
**Output:** `/home/jovyan/work/data/bronze/food_raw` (Delta Lake)

**Pipeline:**
1. Setup SparkSession
2. Baca `food.parquet`
3. Simpan ke Delta Lake + tambah `ingestion_timestamp`
4. Validasi hasil
5. Catat throughput ingesti

## 1. Setup SparkSession

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[2]") \
    .appName("01-bronze-ingestion") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.maxResultSize", "1g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Mode:", spark.sparkContext.master)

Spark version: 3.5.1
Mode: local[2]


## 2. Baca `food.parquet`

In [2]:
df = spark.read.parquet("/home/jovyan/work/food.parquet")

print("Jumlah kolom:", len(df.columns))
print("Schema valid:", df.columns[:5])

Jumlah kolom: 111
Schema valid: ['additives_n', 'additives_tags', 'allergens_tags', 'brands_tags', 'brands']


## 3. Simpan ke Delta Lake

In [3]:
import time
from pyspark.sql.functions import current_timestamp

start = time.time()

df.withColumn("ingestion_timestamp", current_timestamp()) \
  .write \
  .format("delta") \
  .mode("overwrite") \
  .option("maxRecordsPerFile", 250000) \
  .save("/home/jovyan/work/data/bronze/food_raw")

elapsed = time.time() - start
print(f"Bronze Layer tersimpan. Waktu: {elapsed:.1f} detik")

Bronze Layer tersimpan. Waktu: 362.9 detik


## 4. Validasi Hasil

In [4]:
df_bronze = spark.read.format("delta").load("/home/jovyan/work/data/bronze/food_raw")

print("Jumlah baris:", df_bronze.count())
print("Jumlah kolom:", len(df_bronze.columns))
print("Kolom ingestion_timestamp ada:", "ingestion_timestamp" in df_bronze.columns)

Jumlah baris: 4487169
Jumlah kolom: 112
Kolom ingestion_timestamp ada: True


## 5. Catat Throughput Ingesti

In [5]:
throughput = 4487169 / 362.9
print(f"Throughput Bronze: {throughput:,.0f} baris/detik")

Throughput Bronze: 12,365 baris/detik
